In [1]:
# Reload all modules at every code execution
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pfas_trace.config import DATASET_IMPORT, DATASET_FEATURES
from pfas_trace.transform import load_and_transform

from pfas_trace.config import setup_logging

setup_logging()


In [3]:
df = pd.read_parquet(DATASET_IMPORT).query('country=="France"')

In [4]:
df_features = load_and_transform(df)
df_features.head()

2026-11-01 18:39:35,636 :: pfas_trace.transform :: INFO :: unoptimized dataframe size: 546.00 MB
2026-11-01 18:39:35,781 :: pfas_trace.transform :: INFO :: optimized dataframe size: 316.00 MB (-42.03%)


,date,lat,lon,name,city,pfas_sum,matrix,unit,type,sector,...,treatment_and_disposal_of_hazardous_waste_count_5_25km,treatment_and_disposal_of_hazardous_waste_count_25_50km,treatment_and_disposal_of_non_hazardous_waste_distance_min,treatment_and_disposal_of_non_hazardous_waste_count_0_5km,treatment_and_disposal_of_non_hazardous_waste_count_5_25km,treatment_and_disposal_of_non_hazardous_waste_count_25_50km,waste_management_site_distance_min,waste_management_site_count_0_5km,waste_management_site_count_5_25km,waste_management_site_count_25_50km
0,2011-04-27,50.885284,1.662663,BSS000AENC,Wissant,0.0,Groundwater,ng/l,Sampling location,NaN,...,0.0,4.0,47.694805,0.0,0.0,2.0,14.34426,0.0,4.0,4.0
1,2013-06-07,50.885284,1.662663,BSS000AENC,Wissant,0.0,Groundwater,ng/l,Sampling location,NaN,...,0.0,4.0,47.694805,0.0,0.0,2.0,14.34426,0.0,4.0,4.0
2,2015-01-23,50.885284,1.662663,BSS000AENC,Wissant,0.0,Groundwater,ng/l,Sampling location,NaN,...,0.0,4.0,47.694805,0.0,0.0,2.0,14.34426,0.0,4.0,4.0
3,2017-09-12,50.885284,1.662663,BSS000AENC,Wissant,0.0,Groundwater,ng/l,Sampling location,NaN,...,0.0,4.0,47.694805,0.0,0.0,2.0,14.34426,0.0,4.0,4.0
4,2021-02-23,50.885284,1.662663,BSS000AENC,Wissant,0.0,Groundwater,ng/l,Sampling location,NaN,...,0.0,4.0,47.694805,0.0,0.0,2.0,14.34426,0.0,4.0,4.0


In [5]:
# target = pfas_sum
# identifier = date, lat, lon, name, city, matrix, unit
# features: all others
df_features.shape

(734236, 90)

## Description of the feature dataset

The data within the feature dataset ranges the following between the following dates:

In [6]:
print(f"Completedness: {len(df_features) / len(df) * 100:.2f}%")
print(f"Missing city (ignored by model): {df_features["city"].isna().sum()} ({df_features["city"].isnull().sum() / len(df_features) * 100:.2f}%)")
print(f"Missing date: {df_features["date"].isna().sum()}")
print(f"Missing lat/lon: {df_features["lat"].isna().sum() + df_features["lon"].isna().sum()}")
print(f"Missing target: {df_features["pfas_sum"].isna().sum()}")
print("## Temporal range")
print(f"First measurement: {df_features["date"].min().strftime('%d.%m.%Y')}")
print(f"Last measurement: {df_features["date"].max().strftime('%d.%m.%Y')}")

Completedness: 67.10%
Missing city (ignored by model): 338383 (46.09%)
Missing date: 0
Missing lat/lon: 0
Missing target: 0
## Temporal range
First measurement: 04.01.2010
Last measurement: 31.08.2026


### PFAS chemicals

The resulting feature dataset can best be analyzed based on the sample matrix. The zero counts, median values and 99-percentile provide a good overview.

In [7]:
matrix_sizes = df_features["pfas_sum"].groupby(df_features["matrix"]).count().astype(int)
matrix_zeros = df_features["pfas_sum"].astype(int).eq(0).groupby(df_features["matrix"]).sum().astype(int)
matrix_median = df_features["pfas_sum"].groupby(df_features["matrix"]).median()
matrix_max = df_features["pfas_sum"].groupby(df_features["matrix"]).max().astype(int)
matrix_p99 = df_features["pfas_sum"].groupby(df_features["matrix"]).quantile(q=0.99).astype(int)
matrix_unit = df_features["unit"].groupby(df_features["matrix"]).first().astype(str)
pd.DataFrame({
    "total": matrix_sizes,
    "zeros_percent": (matrix_zeros / matrix_sizes) * 100,
    "median": matrix_median,
    "p99": matrix_p99,
    "max": matrix_max,
    "unit": matrix_unit,
})

,total,zeros_percent,median,p99,max,unit
matrix,,,,,,
Biota,3184,30.370603,960.0,42317,1386000,ng/kg fw
Drinking water,46792,87.927423,0.0,139,3798,ng/l
Groundwater,318041,91.175037,0.0,148,4000000,ng/l
Rainwater,165,59.393939,0.0,26159,59000,ng/l
Sediment,13602,80.260256,0.0,55989,1517000,ng/kg dw
Soil,212,0.000000,7890.0,108924,174550,ng/kg
Surface water,339545,53.544891,0.0,690,28000000,ng/l
Suspended matter,534,82.584270,0.0,74699,607000,ng/kg dw
Unknown,144,64.583333,0.0,3406,4300,ng/l


In [8]:
print(f"Totals:")
print(f"zeros: {matrix_zeros.sum()/len(df_features) * 100:.2f}%")
print(f"p99: {df_features["pfas_sum"].quantile(q=0.99):.0f}")
print(f"median: {df_features["pfas_sum"].mean():.0f} max: {df_features["pfas_sum"].max():.0f}")

Totals:
zeros: 72.11%
p99: 1640
median: 2215 max: 509000000


In [9]:
print("Duplicates:")
g = df_features.groupby(["lat", "lon", "date"])
size = g.size()
print(f"Copies: {len(size[size > 1])}, Uniques: {len(size[size <= 1])}")

Duplicates:
Copies: 43269, Uniques: 621334


The following data is describing all the non-zero measurements by year. Please note that a higher percentage does not mean that the contamination got worse! The problematic locations just got sampled more.

In [10]:
nonzero_by_year = df_features["pfas_sum"].gt(0).groupby(df_features["date"].dt.year).sum().astype(int)
round((nonzero_by_year / df_features.groupby(df_features["date"].dt.year).size().astype(int)) * 100, 2)

date
2010     0.80
2011     2.79
2012     2.70
2013     3.03
2014    13.74
2015    20.25
2016    25.14
2017    27.88
2018    27.73
2019    30.08
2020    29.87
2021    30.47
2022    32.70
2023    37.16
2024    40.33
2025    39.26
2026    40.86
dtype: float64

In [11]:
df_features.to_parquet(DATASET_FEATURES)